# Figuras del Capítulo 4 — Compresión SVD

Genera todas las figuras en **castellano** e **inglés** automáticamente.

### Requisitos previos
1. Ejecutar NB02 con la celda de export añadida → genera `results/notebook2/export/`
2. Ejecutar NB03 normalmente → genera `results/notebook3/`
3. Ajustar las rutas en la celda de configuración
4. Poner `tfg_plot_style.py` en el mismo directorio que este notebook

In [ ]:
import sys
import os

from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURACIÓN — Ajusta estas rutas a tu entorno
# ══════════════════════════════════════════════════════════════

# Carpeta de exports generada por la celda añadida a NB02
NB02_EXPORT = PROJECT_ROOT + "results/notebook2/export/"

# Carpeta de resultados de NB03 (los CSVs que ya genera)
NB03_DIR = PROJECT_ROOT + "results/notebook3/"

# ── CSVs de NB02 (generados por la celda de export) ─────────
CSV_K95_MATRIX       = NB02_EXPORT + "rank_matrix_k95.csv"
CSV_SINGULAR_VALUES  = NB02_EXPORT + "singular_values_by_component.csv"
CSV_UNIFORM_RESULTS  = NB02_EXPORT + "uniform_compression_results.csv"
CSV_UNIFORM_EMOTIONS = NB02_EXPORT + "uniform_emotion_f1.csv"
CSV_ADAPTIVE_RANKS   = NB02_EXPORT + "adaptive_rank_distribution.csv"

# ── CSVs de NB03 (nombres reales que ya genera tu notebook) ──
CSV_COMP_SENSITIVITY  = NB03_DIR + "analysis_component_results.csv"
CSV_DEPTH_SENSITIVITY = NB03_DIR + "analysis_depth_results.csv"

# Directorio de salida
import os
os.makedirs(PROJECT_ROOT + "figures", exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tfg_plot_style as sty
sty.apply(lang="es")

def load_csv(path, fallback=None, name=""):
    try:
        df = pd.read_csv(path)
        print(f"✓ {name}: {path} — {df.shape}")
        return df
    except FileNotFoundError:
        if fallback is not None:
            print(f"⚠ {name}: no encontrado — usando fallback")
            return fallback
        print(f"✗ {name}: {path} — NO ENCONTRADO")
        return None

---
## Fig. 1 — Heatmap de $k_{95}$ (anatomía arquitectónica)

In [ ]:
k95_df = load_csv(CSV_K95_MATRIX, name="k95 matrix")

if k95_df is not None:
    def plot_anatomy(ax, df=k95_df):
        df = df.set_index(df.columns[0])  # first col = layer index
        data = df.values.astype(float)
        comp_names = list(df.columns)
        n_layers = data.shape[0]

        im = ax.imshow(data, cmap=sty.CMAP_COOL, aspect="auto", vmin=150, vmax=700)
        ax.set_xticks(range(len(comp_names)))
        ax.set_xticklabels(comp_names, fontsize=sty.TICK_SIZE)
        ax.set_yticks(range(n_layers))
        ax.set_yticklabels([f"L{i}" for i in range(n_layers)])
        ax.set_ylabel(sty.L["layer"])
        ax.set_xlabel(sty.L["component"])
        ax.set_title(sty.L["t_anatomy"])

        for i in range(n_layers):
            for j in range(len(comp_names)):
                val = int(data[i, j])
                color = "white" if val > 500 else sty.INK
                ax.text(j, i, str(val), ha="center", va="center",
                        fontsize=sty.SMALL_SIZE, color=color)

        ffn_idx = next((i for i, c in enumerate(comp_names) if "FFN" in c or "ffn" in c), 4)
        ax.axvline(x=ffn_idx - 0.5, color=sty.INK, lw=1.5, ls="--", alpha=0.4)
        ax.text((ffn_idx - 1) / 2, -0.8, sty.L["self_attention"], ha="center",
                fontsize=sty.ANNOTATION_SIZE, color=sty.INK_3, style="italic")
        ax.text((ffn_idx + len(comp_names) - 1) / 2, -0.8, sty.L["feed_forward"], ha="center",
                fontsize=sty.ANNOTATION_SIZE, color=sty.INK_3, style="italic")

        cbar = ax.figure.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
        cbar.set_label("$k_{95}$", fontsize=sty.LABEL_SIZE)

    sty.generate_both(plot_anatomy, "anatomy_heatmap", figsize=sty.FIG_FULL)
else:
    print("⏭ Ejecuta NB02 con la celda de export para generar rank_matrix_k95.csv")

---
## Fig. 2 — Decaimiento espectral por componente

In [ ]:
sv_df = load_csv(CSV_SINGULAR_VALUES, name="singular values")

if sv_df is not None:
    def plot_spectral(ax, df=sv_df):
        # Columnas: rank_index, Query_Q_mean, Query_Q_std, Key_K_mean, ...
        mean_cols = [c for c in df.columns if c.endswith('_mean')]
        x = df["rank_index"].values if "rank_index" in df.columns else np.arange(1, len(df)+1)

        for col in mean_cols:
            comp_name = col.replace('_mean', '').replace('_', ' ')
            std_col = col.replace('_mean', '_std')
            color = sty.component_color(comp_name)
            lw = 2.0 if "FFN" in comp_name else 1.5
            ls = "-" if any(k in comp_name for k in ("Query", "Key", "FFN")) else "--"

            ax.plot(x, df[col], color=color, lw=lw, ls=ls, label=comp_name, alpha=0.85)
            if std_col in df.columns:
                ax.fill_between(x, df[col] - df[std_col], df[col] + df[std_col],
                                color=color, alpha=0.08)

        ax.set_xlabel(sty.L["singular_index"])
        ax.set_ylabel(sty.L["singular_magnitude"])
        ax.set_title(sty.L["t_spectral"])
        ax.legend(loc="upper right", ncol=2)
        ax.set_xlim(0, None)
        ax.set_ylim(-0.02, 1.05)

    sty.generate_both(plot_spectral, "spectral_fingerprint", figsize=sty.FIG_FULL)
else:
    print("⏭ Ejecuta NB02 con la celda de export para generar singular_values_by_component.csv")

---
## Fig. 3 — Compresión uniforme: transición de fase

In [ ]:
uni_fallback = pd.DataFrame({
    "rank":      [768,   512,   384,   256,   128,   64],
    "macro_f1":  [0.577, 0.464, 0.251, 0.025, 0.000, 0.000],
    "retention": [1.000, 0.805, 0.435, 0.043, 0.000, 0.000],
})
uni_df = load_csv(CSV_UNIFORM_RESULTS, fallback=uni_fallback, name="uniform results")

def plot_transition(ax, df=uni_df):
    ranks = df["rank"].values
    retention = df["retention"].values

    ax.plot(ranks, retention, color=sty.BLUE, marker="s", markersize=8, lw=2.2,
            markeredgecolor="white", markeredgewidth=1.2, zorder=5)

    ax.axvspan(0, 256, alpha=0.04, color=sty.TERRA, zorder=0)
    ax.text(160, 0.88, sty.L["collapse_zone"], ha="center",
            fontsize=sty.ANNOTATION_SIZE, color=sty.TERRA_L, style="italic")
    ax.axvspan(256, 384, alpha=0.04, color=sty.SAND, zorder=0)
    ax.annotate(sty.L["phase_transition"], xy=(320, 0.24),
                fontsize=sty.ANNOTATION_SIZE, color=sty.INK_3, ha="center", style="italic")

    for r, ret, label in [(512, 0.805, "80.5%"), (384, 0.435, "43.5%"), (256, 0.043, "4.3%")]:
        offset = (10, 10) if ret > 0.1 else (10, -15)
        sty.annotate_point(ax, r, ret, label, offset=offset)

    ax.set_xlabel(sty.L["rank"])
    ax.set_ylabel(sty.L["f1_retention"])
    ax.set_title(sty.L["t_uniform"])
    ax.set_xlim(0, 800)
    ax.set_ylim(-0.05, 1.1)
    sty.format_pct(ax)
    ax.invert_xaxis()

sty.generate_both(plot_transition, "uniform_transition", figsize=sty.FIG_WIDE)

---
## Fig. 4 — Vulnerabilidad emocional bajo compresión uniforme

In [ ]:
emo_df = load_csv(CSV_UNIFORM_EMOTIONS, name="uniform emotion F1")

if emo_df is not None:
    def plot_emotion_vuln(ax, df=emo_df):
        # Set emotion as index if present
        if df.columns[0].lower() in ('emotion', 'emocion'):
            df = df.set_index(df.columns[0])
        # Sort by first column (baseline) ascending so highest F1 is at top
        df = df.sort_values(df.columns[0], ascending=True)

        im = ax.imshow(df.values, cmap=sty.CMAP_SEQUENTIAL, aspect="auto", vmin=0, vmax=1)
        ax.set_xticks(range(len(df.columns)))
        ax.set_xticklabels(df.columns, fontsize=sty.TICK_SIZE, rotation=45, ha="right")
        ax.set_yticks(range(len(df.index)))
        ax.set_yticklabels(df.index, fontsize=7)
        ax.set_title(sty.L["t_emotion_vuln"])

        for i in range(len(df.index)):
            for j in range(len(df.columns)):
                val = df.iloc[i, j]
                if not np.isnan(val) and val > 0.005:
                    color = "white" if val > 0.5 else sty.INK
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                            fontsize=5.5, color=color)

        cbar = ax.figure.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
        cbar.set_label("F1", fontsize=sty.LABEL_SIZE)

    sty.generate_both(plot_emotion_vuln, "emotion_vulnerability", figsize=sty.FIG_TALL)
else:
    print("⏭ Ejecuta NB02 con la celda de export para generar uniform_emotion_f1.csv")

---
## Fig. 5 — Sensibilidad por componente a $r=128$

Lee `analysis_component_results.csv` de NB03 y filtra por `rank==128`.

In [ ]:
comp_raw = load_csv(CSV_COMP_SENSITIVITY, name="NB03 component sensitivity")

if comp_raw is not None:
    # NB03 format: component, rank, f1_macro, compression_ratio, n_layers, f1_retention, silhouette
    # Filter to rank=128 for the main bar chart
    baseline_f1 = comp_raw.loc[comp_raw['f1_retention'].idxmax(), 'f1_macro'] / (comp_raw['f1_retention'].max() / 100) if 'f1_retention' in comp_raw.columns else 0.577
    comp_128 = comp_raw[comp_raw['rank'] == 128].copy()
    comp_128['retention'] = comp_128['f1_retention'] / 100  # convert from pct to fraction
    comp_128 = comp_128.sort_values('retention', ascending=False)
    print(f"  Componentes a r=128: {len(comp_128)}")
    print(comp_128[['component', 'f1_macro', 'retention']].to_string(index=False))
else:
    # Fallback
    comp_128 = pd.DataFrame({
        "component": ["Query (Q)", "Key (K)", "Value (V)", "Attn Output", "FFN Output", "FFN Inter"],
        "f1_macro":  [0.574, 0.568, 0.395, 0.325, 0.128, 0.040],
        "retention": [0.994, 0.984, 0.685, 0.563, 0.222, 0.069],
    })

In [ ]:
def plot_comp_sens(ax, df=comp_128):
    comps = df["component"].values
    retentions = df["retention"].values
    colors = [sty.component_color(c) for c in comps]

    bars = ax.barh(range(len(comps)), retentions, color=colors, height=0.55,
                   edgecolor=sty.BG, linewidth=0.8)

    for i, (comp, ret) in enumerate(zip(comps, retentions)):
        x_pos = ret + 0.015 if ret < 0.85 else ret - 0.07
        color = sty.INK_2 if ret < 0.85 else "white"
        ax.text(x_pos, i, f"{ret:.1%}", va="center", fontsize=sty.ANNOTATION_SIZE,
                color=color, weight="medium")

    ax.set_yticks(range(len(comps)))
    ax.set_yticklabels(comps)
    ax.set_xlabel(sty.L["f1_retention"])
    ax.set_title(sty.L["t_comp_sens"])
    ax.set_xlim(0, 1.12)
    sty.format_pct(ax, axis="x")
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.3)
    ax.grid(axis="y", visible=False)
    ax.axvline(x=0.80, color=sty.INK_3, ls=":", lw=0.7, alpha=0.5)

sty.generate_both(plot_comp_sens, "component_sensitivity", figsize=sty.FIG_WIDE)

---
## Fig. 6 — Sensibilidad por grupo de profundidad

Lee `analysis_depth_results.csv` de NB03 y pivotea por rango.

In [ ]:
depth_raw = load_csv(CSV_DEPTH_SENSITIVITY, name="NB03 depth sensitivity")

if depth_raw is not None:
    # NB03 format: depth_group, rank, f1_macro, f1_retention, silhouette
    # Pivot to: group | r256 | r128 | r64
    depth_raw['retention'] = depth_raw['f1_retention'] / 100
    depth_pivot = depth_raw.pivot(index='depth_group', columns='rank', values='retention')
    depth_pivot.columns = [f'r{int(c)}' for c in depth_pivot.columns]
    depth_pivot = depth_pivot.reset_index()
    depth_pivot.columns = ['group'] + list(depth_pivot.columns[1:])
    
    # Add line breaks for display
    group_map = {
        'Early (0-3)': 'Early\n(0–3)', 'Middle (4-7)': 'Middle\n(4–7)', 'Late (8-11)': 'Late\n(8–11)',
        'early': 'Early\n(0–3)', 'middle': 'Middle\n(4–7)', 'late': 'Late\n(8–11)',
    }
    depth_pivot['group'] = depth_pivot['group'].map(lambda g: group_map.get(g, g))
    print(depth_pivot)
else:
    depth_pivot = pd.DataFrame({
        "group": ["Early\n(0–3)", "Middle\n(4–7)", "Late\n(8–11)"],
        "r256":  [0.854, 0.825, 0.333],
        "r128":  [0.625, 0.465, 0.000],
        "r64":   [0.370, 0.253, 0.000],
    })

In [ ]:
def plot_depth_sens(ax, df=depth_pivot):
    groups = df["group"].values
    x = np.arange(len(groups))
    rank_cols = [c for c in df.columns if c.startswith('r') and c != 'group']
    width = 0.7 / len(rank_cols)
    rank_colors = [sty.SAGE, sty.TERRA_L, sty.TERRA][:len(rank_cols)]

    for i, (col, rc) in enumerate(zip(rank_cols, rank_colors)):
        vals = df[col].values
        label = f"$r{{=}}{col.replace('r', '')}$"
        ax.bar(x + i * width, vals, width, label=label, color=rc,
               edgecolor=sty.BG, linewidth=0.8)
        for j, v in enumerate(vals):
            text = f"{v:.0%}" if v > 0.01 else "0%"
            color = sty.INK_3 if v > 0.01 else sty.TERRA
            y_pos = v + 0.02 if v > 0.01 else 0.02
            ax.text(x[j] + i * width, y_pos, text, ha="center",
                    fontsize=sty.SMALL_SIZE, color=color)

    ax.set_xticks(x + width * (len(rank_cols) - 1) / 2)
    ax.set_xticklabels(groups)
    ax.set_ylabel(sty.L["f1_retention"])
    ax.set_title(sty.L["t_depth_sens"])
    ax.set_ylim(0, 1.05)
    sty.format_pct(ax)
    ax.legend(fontsize=sty.LEGEND_SIZE)

sty.generate_both(plot_depth_sens, "depth_sensitivity", figsize=sty.FIG_WIDE)

---
## Fig. 7 — Frontera de Pareto

Combina los resultados de NB02 (`spectral_analysis_results.csv`) con los de NB03 si están disponibles.

In [ ]:
# Try NB02's main results CSV
pareto_csv = NB02_EXPORT.replace('export/', '') + "spectral_analysis_results.csv"
pareto_df = load_csv(pareto_csv, name="NB02 strategy results")

if pareto_df is not None:
    # Infer strategy type from name
    def infer_type(name):
        n = name.lower()
        if 'uniform' in n or n.startswith('r=') or n.startswith('r '): return 'uniform'
        if 'adaptive' in n or 'adapt' in n: return 'adaptive'
        if 'mixed' in n or 'conserv' in n or 'aggress' in n: return 'adaptive'
        if 'baseline' in n: return 'baseline'
        return 'uniform'
    
    pareto_df['type'] = pareto_df['strategy'].apply(infer_type)
    # Normalize retention to 0-1 range
    if pareto_df['f1_retention'].max() > 2:  # it's in percentage
        pareto_df['f1_retention'] = pareto_df['f1_retention'] / 100
    print(pareto_df[['strategy', 'type', 'compression_ratio', 'f1_retention']])
else:
    pareto_df = pd.DataFrame({
        "strategy": ["uniform_r512", "uniform_r384", "uniform_r256", "uniform_r128",
                     "adaptive_e99", "adaptive_e95", "adaptive_e90", "adaptive_e80"],
        "type":     ["uniform"]*4 + ["adaptive"]*4,
        "compression_ratio": [1.0, 0.806, 0.612, 0.418, 1.202, 1.022, 0.892, 0.721],
        "f1_retention":      [0.805, 0.435, 0.043, 0.0, 0.967, 0.850, 0.619, 0.248],
    })

In [ ]:
def plot_pareto(ax, df=pareto_df):
    for stype in [t for t in df["type"].unique() if t != 'baseline']:
        subset = df[df["type"] == stype].sort_values("compression_ratio")
        color = sty.strategy_color(stype)
        marker = sty.strategy_marker(stype)
        label = sty.L.get(stype, stype.capitalize())

        ax.plot(subset["compression_ratio"], subset["f1_retention"],
                color=color, lw=1.0, alpha=0.4, zorder=3)
        ax.scatter(subset["compression_ratio"], subset["f1_retention"],
                  color=color, marker=marker, s=70, edgecolor="white",
                  linewidth=0.8, zorder=5, label=label)

    ax.axhline(y=1.0, color=sty.INK_3, ls=":", lw=0.6, alpha=0.4)
    ax.axvline(x=1.0, color=sty.INK_3, ls=":", lw=0.6, alpha=0.4)
    ax.text(1.02, 0.02, sty.L["baseline_params"], fontsize=sty.SMALL_SIZE,
            color=sty.INK_3, style="italic")

    xmax = df["compression_ratio"].max() + 0.05
    if xmax > 1.05:
        ax.axvspan(1.0, xmax, alpha=0.03, color=sty.TERRA)
        ax.text((1.0 + xmax) / 2, 0.92, sty.L["expansion_zone"],
                fontsize=sty.SMALL_SIZE, color=sty.INK_3, ha="center", style="italic")

    ax.legend(loc="lower right")
    ax.set_xlabel(sty.L["param_ratio"])
    ax.set_ylabel(sty.L["f1_retention"])
    ax.set_title(sty.L["t_pareto"])
    ax.set_ylim(-0.05, 1.1)
    sty.format_pct(ax)

sty.generate_both(plot_pareto, "pareto_frontier", figsize=sty.FIG_FULL)

---
## Fig. 8 — Distribución de rangos adaptativos por umbral

In [ ]:
arank_df = load_csv(CSV_ADAPTIVE_RANKS, name="adaptive ranks")

if arank_df is not None:
    def plot_adaptive_ranks(ax, df=arank_df):
        thresholds = sorted(df["threshold"].unique())
        components = df["component"].unique()

        x = np.arange(len(thresholds))
        n = len(components)
        w = 0.8 / n

        for i, comp in enumerate(components):
            sub = df[df["component"] == comp].sort_values("threshold")
            color = sty.component_color(comp)
            ax.bar(x + i*w - (n-1)*w/2, sub["mean_rank"].values, w,
                   label=comp.replace("_", " "), color=color, edgecolor=sty.BG, linewidth=0.5)

        ax.axhline(y=384, color=sty.INK_3, ls="--", lw=0.7, alpha=0.5)
        ax.text(len(thresholds)-0.3, 390, sty.L["break_even_attn"],
                fontsize=sty.SMALL_SIZE, color=sty.INK_3, va="bottom")
        ax.axhline(y=614, color=sty.INK_3, ls="--", lw=0.7, alpha=0.5)
        ax.text(len(thresholds)-0.3, 620, sty.L["break_even_ffn"],
                fontsize=sty.SMALL_SIZE, color=sty.INK_3, va="bottom")

        ax.set_xticks(x)
        ax.set_xticklabels([f"$\\tau$={t}%" for t in thresholds])
        ax.set_ylabel(sty.L["rank_assigned"])
        ax.set_title(sty.L["t_adaptive_ranks"])
        ax.set_ylim(0, 780)
        ax.legend(fontsize=sty.SMALL_SIZE, ncol=3, loc="upper left")
        ax.grid(axis="x", visible=False)

    sty.generate_both(plot_adaptive_ranks, "adaptive_rank_distribution", figsize=sty.FIG_WIDE)
else:
    print("⏭ Ejecuta NB02 con la celda de export para generar adaptive_rank_distribution.csv")

---
## Resumen

| # | Archivo | § | Origen |
|---|---------|---|--------|
| 1 | `cap4_anatomy_heatmap_{es,en}` | §4.1 | NB02 export: `rank_matrix_k95.csv` |
| 2 | `cap4_spectral_fingerprint_{es,en}` | §4.1 | NB02 export: `singular_values_by_component.csv` |
| 3 | `cap4_uniform_transition_{es,en}` | §4.2 | NB02 export: `uniform_compression_results.csv` (o fallback) |
| 4 | `cap4_emotion_vulnerability_{es,en}` | §4.2 | NB02 export: `uniform_emotion_f1.csv` |
| 5 | `cap4_component_sensitivity_{es,en}` | §4.3 | NB03: `analysis_component_results.csv` |
| 6 | `cap4_depth_sensitivity_{es,en}` | §4.3 | NB03: `analysis_depth_results.csv` |
| 7 | `cap4_pareto_frontier_{es,en}` | §4.4 | NB02: `spectral_analysis_results.csv` (o fallback) |
| 8 | `cap4_adaptive_rank_distribution_{es,en}` | §4.4 | NB02 export: `adaptive_rank_distribution.csv` |